In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "knoveleng/Open-RS1"

print("Đang tải tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tải bản sao thứ nhất lên GPU 0
print("Đang tải model bản sao 1 lên GPU 0 (cuda:0)...")
model_0 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Bắt buộc dùng float16 cho card T4
    device_map={'': 'cuda:0'}
)

# Tải bản sao thứ hai lên GPU 1
print("Đang tải model bản sao 2 lên GPU 1 (cuda:1)...")
model_1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={'': 'cuda:1'}
)

print("Tải model 1 hoàn tất! Sẵn sàng cho 2 GPU.")

Đang tải tokenizer...


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Đang tải model bản sao 2 lên GPU 1 (cuda:1)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Tải model 1 hoàn tất! Sẵn sàng cho 2 GPU.


In [ ]:
import pandas as pd

# Đường dẫn tới file jsonl của bạn
file_path = "/kaggle/input/datasets/qucnguyntin/multiarith/MultiArith_eval_ready.jsonl"
df = pd.read_json(file_path, lines=True)


# Đảm bảo số lượng câu hỏi luôn là số chẵn để chia đều cho 2 GPU
if len(df) % 2 != 0:
    df = df.iloc[:-1]

questions = df['Question'].tolist()
ground_truths = df['Answer'].tolist()

print(f"Đã tải xong dữ liệu")

Đã tải xong dữ liệu


In [3]:
import re
import concurrent.futures
from tqdm.notebook import tqdm

# Ép model luôn bỏ kết quả cuối cùng vào \boxed{} để dùng Regex tìm
system_prompt = r"""Solve the math problem with clear, brief, and logical steps. Do not overthink or add unnecessary text. Put your final answer within \boxed{}
Example:
Question: Nancy had 41 pictures. She put 37 pics into one album and put the rest into 2 different albums. How many pictures were in each album?
Answer: 
- Total pictures initially: 41
- Pictures left after first album: 41 - 37 = 4
- Divide the rest into 2 albums: 4 / 2 = 2
- Pictures in each of the other albums: 2
\boxed{2}

Now solve this:
"""

# Hàm chạy suy luận trên một GPU cụ thể
def process_on_gpu(question, model, device):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.3,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    # Lấy phần text mới được sinh ra
    response_ids = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

# Hàm dùng Regex để tìm đáp án trong \boxed{}
def extract_answer(text):
    match = re.search(r"\\boxed\{([^}]*)\}", text)
    if match:
        return match.group(1).strip()
    return None

correct_count = 0
total_count = len(questions)
failed_example = None


# Vòng lặp nhảy bước 2 (để lấy 2 câu mỗi lần)
for i in tqdm(range(0, total_count, 2)):
    q1 = questions[i]
    q2 = questions[i+1]
    
    # Mở 2 luồng (thread) để bắt 2 GPU làm việc cùng một lúc
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        future_0 = executor.submit(process_on_gpu, q1, model_0, "cuda:0")
        future_1 = executor.submit(process_on_gpu, q2, model_1, "cuda:1")
        
        # Chờ cả 2 GPU trả về kết quả
        ans_text_1 = future_0.result()
        ans_text_2 = future_1.result()

    # Đánh giá câu 1
    extracted_1 = extract_answer(ans_text_1)
    truth_1 = str(ground_truths[i]).strip()
    if extracted_1 == truth_1:
        correct_count += 1
    elif failed_example is None: # Lưu lại câu sai đầu tiên
        failed_example = (q1, truth_1, extracted_1, ans_text_1)

    # Đánh giá câu 2
    extracted_2 = extract_answer(ans_text_2)
    truth_2 = str(ground_truths[i+1]).strip()
    if extracted_2 == truth_2:
        correct_count += 1
    elif failed_example is None:
        failed_example = (q2, truth_2, extracted_2, ans_text_2)

# ==========================================
# IN KẾT QUẢ ĐÚNG YÊU CẦU
# ==========================================
success_rate = (correct_count / total_count) * 100
print("\n" + "="*50)
print(f"Tỉ lệ thành công: {success_rate:.2f}%")
print("="*50)

if failed_example:
    q, truth, ext, full_ans = failed_example
    print("\n=== MỘT CÂU TRẢ LỜI SAI MẪU ===")
    print(f"[Câu hỏi]:\n{q}")
    print("-" * 30)
    print(f"[Kết quả thực sự]: {truth}")
    print(f"[Kết quả Model đã đưa ra (trích xuất được)]: {ext}")
    print("-" * 30)
    print("[Toàn bộ lập luận của Model]:\n")
    print(full_ans)

  0%|          | 0/210 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Tỉ lệ thành công: 97.62%

=== MỘT CÂU TRẢ LỜI SAI MẪU ===
[Câu hỏi]:
 Sam invited 9 friends to a birthday party, but 6 couldn't come. If he wanted to buy enough cupcakes so each person could have exactly 2, how many should he buy? 
------------------------------
[Kết quả thực sự]: 6
[Kết quả Model đã đưa ra (trích xuất được)]: 8
------------------------------
[Toàn bộ lập luận của Model]:

<think>
First, I need to determine how many friends actually attended the party. Sam invited 9 friends, and 6 couldn't come, so 9 minus 6 equals 3 friends attended.

Next, I'll calculate the total number of people at the party, including Sam, by adding the 3 attendees to the 1 person who was there to begin with. This gives a total of 4 people.

Since Sam wants each person to have exactly 2 cupcakes, I'll multiply the total number of people by 2. That means 4 people multiplied by 2 cupcakes each equals 8 cupcakes needed.

Therefore, Sam should buy 8 cupcakes to ensure each person has exactly 2.
</thi